# 2026年塔什干国际奥林匹克数学竞赛Q1

## 问题描述

在一个无限方格网格上，一辆汽车占据一个单位方格，朝向为东、南、西或北。两名玩家轮流操作：

    玩家1：将汽车沿当前朝向向前移动至少1格、至多10格。

    玩家2：将汽车原地左转90°或右转90°。

    玩家1先手。如果在前1000次玩家1的移动中，汽车访问了某个方格两次，则玩家2获胜。否则玩家1获胜。确定哪位玩家有必胜策略。

    注：玩家2旋转汽车或驾车穿过方格都不算新的访问，特别地，玩家1的每次移动恰好使一个方格的访问次数增加1。

## 必胜策略

 这个游戏，玩家2将拥有必胜策略，策略如下：

    轮到玩家2行动的时候。

    如果是奇数回合，汽车落在第一，二象限，那么玩家2要使得汽车朝向下，否则朝上。

    如果是偶数回合，汽车落在第一，四象限，那么玩家2要使得汽车朝向左，否则朝右。

    这种迫使汽车总是在原点附近往返移动的方法，称为原点回归策略。



编写python代码，根据这个必胜策略，多次模拟游戏进行，统计玩家2的胜率，并显示第一次游戏的全局进程

In [6]:
import random
from collections import defaultdict

# 方向编码: 0=东, 1=北, 2=西, 3=南
DIR_VECTORS = [(1, 0), (0, 1), (-1, 0), (0, -1)]
DIR_NAMES = ['东', '北', '西', '南']

def rotate_left(d):
    return (d + 1) % 4

def rotate_right(d):
    return (d - 1) % 4

def player2_action(current_dir, step, x, y):
    """原点回归策略"""
    if step % 2 == 1:          # 奇数回合
        target = 3 if y > 0 else 1   # 南或北
    else:                      # 偶数回合
        target = 2 if x > 0 else 0   # 西或东

    # 旋转到目标方向（一定可达）
    if rotate_left(current_dir) == target:
        return target, '左转'
    elif rotate_right(current_dir) == target:
        return target, '右转'
    else:
        # 如果当前已经是目标方向，无需旋转
        return current_dir, '不变'

def simulate_with_trace(initial_dir=0, max_steps=1000, player1_strategy='avoid'):
    """模拟一局并打印详细进程"""
    x, y = 0, 0
    direction = initial_dir
    # visited 现在是字典：坐标 -> 首次访问的步数（0表示初始位置）
    visited = {(0, 0): 0}
    print(f"初始状态: 位置 (0,0), 朝向 {DIR_NAMES[direction]}，方格 (0,0) 在第0步已访问")
    print("-" * 70)

    for step in range(1, max_steps + 1):
        # 玩家1选择移动距离
        if player1_strategy == 'random':
            dist = random.randint(1, 10)
            choice_reason = "随机"
        else:  # avoid 策略
            safe_moves = []
            for d in range(1, 11):
                nx = x + DIR_VECTORS[direction][0] * d
                ny = y + DIR_VECTORS[direction][1] * d
                if (nx, ny) not in visited:
                    safe_moves.append(d)
            if safe_moves:
                dist = random.choice(safe_moves)
                choice_reason = f"避免重复，可选安全距离 {safe_moves}，选择 {dist}"
            else:
                dist = random.randint(1, 10)
                choice_reason = "所有可达方格均已访问，随机选择"

        # 移动
        new_x = x + DIR_VECTORS[direction][0] * dist
        new_y = y + DIR_VECTORS[direction][1] * dist
        print(f"第{step}步(玩家1): 从 ({x},{y}) 向 {DIR_NAMES[direction]} 移动 {dist} 格 → ({new_x},{new_y}) [{choice_reason}]")

        # 检查重复
        if (new_x, new_y) in visited:
            first_step = visited[(new_x, new_y)]
            if first_step == 0:
                print(f"★ 方格 ({new_x},{new_y}) 是起点，在第0步已访问过！玩家2获胜。游戏结束。")
            else:
                print(f"★ 方格 ({new_x},{new_y}) 已在第 {first_step} 步首次访问过！玩家2获胜。游戏结束。")
            return 'player2', step
        # 记录本次访问的步数
        visited[(new_x, new_y)] = step
        x, y = new_x, new_y

        # 玩家2旋转
        new_dir, rot_name = player2_action(direction, step, x, y)
        print(f"       玩家2: 当前回合{'奇数' if step%2==1 else '偶数'}，位置 ({x},{y}) -> 目标 {DIR_NAMES[new_dir]}，执行 {rot_name}")
        print(f"       结果朝向: {DIR_NAMES[new_dir]}")
        direction = new_dir
        print("-" * 70)

    return 'player1', max_steps

def simulate(initial_dir=0, max_steps=1000, player1_strategy='avoid'):
    """安静模拟，用于批量统计"""
    x, y = 0, 0
    direction = initial_dir
    visited = {(0, 0)}  # 统计时只需 set 就够了

    for step in range(1, max_steps + 1):
        if player1_strategy == 'random':
            dist = random.randint(1, 10)
        else:
            safe_moves = []
            for d in range(1, 11):
                nx = x + DIR_VECTORS[direction][0] * d
                ny = y + DIR_VECTORS[direction][1] * d
                if (nx, ny) not in visited:
                    safe_moves.append(d)
            dist = random.choice(safe_moves) if safe_moves else random.randint(1, 10)

        x += DIR_VECTORS[direction][0] * dist
        y += DIR_VECTORS[direction][1] * dist
        if (x, y) in visited:
            return 'player2', step
        visited.add((x, y))

        direction, _ = player2_action(direction, step, x, y)

    return 'player1', max_steps

def run_trials(num_games=5000, initial_dir=0, strategy='avoid'):
    wins = defaultdict(int)
    step_counts = []
    for _ in range(num_games):
        winner, steps = simulate(initial_dir=initial_dir, player1_strategy=strategy)
        wins[winner] += 1
        step_counts.append(steps)
    return wins, step_counts

if __name__ == '__main__':
    random.seed(42)
    
    print("===== 第一次游戏的全局进程 =====")
    winner, steps = simulate_with_trace(initial_dir=0, player1_strategy='avoid')
    print(f"\n本次游戏结果: {winner} 在第 {steps} 步获胜。\n")

    print("===== 批量模拟统计 =====")
    trials = 5000
    wins, step_list = run_trials(num_games=trials, initial_dir=0, strategy='avoid')
    print(f"玩家2获胜: {wins['player2']} 次 ({100 * wins['player2'] / trials:.2f}%)")
    print(f"玩家1获胜: {wins['player1']} 次 ({100 * wins['player1'] / trials:.2f}%)")
    if step_list:
        print(f"玩家2获胜平均步数: {sum(step_list)/len(step_list):.1f}，最大步数: {max(step_list)}")

===== 第一次游戏的全局进程 =====
初始状态: 位置 (0,0), 朝向 东，方格 (0,0) 在第0步已访问
----------------------------------------------------------------------
第1步(玩家1): 从 (0,0) 向 东 移动 2 格 → (2,0) [避免重复，可选安全距离 [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]，选择 2]
       玩家2: 当前回合奇数，位置 (2,0) -> 目标 北，执行 左转
       结果朝向: 北
----------------------------------------------------------------------
第2步(玩家1): 从 (2,0) 向 北 移动 1 格 → (2,1) [避免重复，可选安全距离 [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]，选择 1]
       玩家2: 当前回合偶数，位置 (2,1) -> 目标 西，执行 左转
       结果朝向: 西
----------------------------------------------------------------------
第3步(玩家1): 从 (2,1) 向 西 移动 5 格 → (-3,1) [避免重复，可选安全距离 [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]，选择 5]
       玩家2: 当前回合奇数，位置 (-3,1) -> 目标 南，执行 左转
       结果朝向: 南
----------------------------------------------------------------------
第4步(玩家1): 从 (-3,1) 向 南 移动 4 格 → (-3,-3) [避免重复，可选安全距离 [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]，选择 4]
       玩家2: 当前回合偶数，位置 (-3,-3) -> 目标 东，执行 左转
       结果朝向: 东
--------------------------------------------------------------------

用强化学习让程序自己训练出这个必胜策略，

给出对应python代码：

要求先训练出这个策略，然后这个输入汽车坐标的时候，以及汽车朝向的时候，策略可以告诉我玩家2应该如何决策

In [8]:
import random
from collections import defaultdict

# =================== 环境与基础定义 ===================
# 方向: 0=东, 1=北, 2=西, 3=南
DIR_VECTORS = [(1, 0), (0, 1), (-1, 0), (0, -1)]
DIR_NAMES = ['东', '北', '西', '南']

def rotate_left(d):
    return (d + 1) % 4

def rotate_right(d):
    return (d - 1) % 4

class Game:
    """游戏环境，只负责状态更新，玩家1移动策略由外部传入"""
    def __init__(self, max_steps=1000):
        self.max_steps = max_steps

    def reset(self):
        self.x = 0
        self.y = 0
        self.direction = 0  # 初始朝东
        self.visited = {(0, 0)}
        self.step = 1  # 将要执行第1次玩家1移动
        return self._get_state()

    def _get_state(self):
        """返回当前状态（玩家2做决策前的状态）"""
        return (self.x, self.y, self.direction, self.step)

    def player1_move(self, dist):
        """玩家1移动指定距离，返回 (新状态, 是否重复, 是否结束)"""
        dx, dy = DIR_VECTORS[self.direction]
        self.x += dx * dist
        self.y += dy * dist
        if (self.x, self.y) in self.visited:
            return self._get_state(), True, True
        self.visited.add((self.x, self.y))
        self.step += 1
        if self.step > self.max_steps:
            return self._get_state(), False, True
        return self._get_state(), False, False

    def player2_rotate(self, action):
        """玩家2旋转: 0=左转, 1=右转"""
        if action == 0:
            self.direction = rotate_left(self.direction)
        else:
            self.direction = rotate_right(self.direction)

# =================== 特征提取 ===================
def extract_features(state):
    """从原始状态提取用于Q学习的离散特征"""
    x, y, direction, step = state
    # x>0 为1, 否则0; y>0 为1, 否则0
    sign_x = 1 if x > 0 else 0
    sign_y = 1 if y > 0 else 0
    parity = step % 2  # 奇偶性
    return (sign_x, sign_y, direction, parity)

# =================== 玩家1移动策略 ===================
def random_p1_move(env):
    """玩家1随机移动1~10格"""
    return random.randint(1, 10)

def avoid_p1_move(env):
    """玩家1优先选择不重复的移动距离"""
    safe = []
    for d in range(1, 11):
        nx = env.x + DIR_VECTORS[env.direction][0] * d
        ny = env.y + DIR_VECTORS[env.direction][1] * d
        if (nx, ny) not in env.visited:
            safe.append(d)
    return random.choice(safe) if safe else random.randint(1, 10)

# =================== Q-learning 智能体 ===================
class QLearningAgent:
    def __init__(self, alpha=0.1, gamma=1.0, epsilon=0.1):
        self.Q = defaultdict(lambda: [0.0, 0.0])  # 两个动作: 0左转, 1右转
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = epsilon

    def get_action(self, state, greedy=False):
        """返回动作索引，greedy=True时只选最优"""
        f_state = extract_features(state)
        if not greedy and random.random() < self.epsilon:
            return random.randint(0, 1)
        q_values = self.Q[f_state]
        if q_values[0] > q_values[1]:
            return 0
        elif q_values[1] > q_values[0]:
            return 1
        else:
            return random.randint(0, 1)  # 平局随机

    def update(self, state, action, reward, next_state, done):
        f_state = extract_features(state)
        f_next = extract_features(next_state)
        q_next_max = 0.0 if done else max(self.Q[f_next])
        target = reward + self.gamma * q_next_max
        td_error = target - self.Q[f_state][action]
        self.Q[f_state][action] += self.alpha * td_error

# =================== 训练循环 ===================
def train(episodes=50000, p1_strategy='random'):
    env = Game(max_steps=1000)
    agent = QLearningAgent(alpha=0.2, gamma=1.0, epsilon=0.3)
    # 随着训练逐渐减小 epsilon
    epsilon_start = 0.3
    epsilon_end = 0.01
    epsilon_decay = (epsilon_start - epsilon_end) / episodes

    p1_move = random_p1_move if p1_strategy == 'random' else avoid_p1_move

    for ep in range(episodes):
        state = env.reset()
        agent.epsilon = max(epsilon_end, epsilon_start - ep * epsilon_decay)
        done = False

        while not done:
            # 玩家1移动
            dist = p1_move(env)
            next_state, repeated, done = env.player1_move(dist)

            if repeated:
                # 玩家2胜利，获得奖励
                reward = 1.0
                # 最后一个状态（玩家2无需再动作），直接更新
                agent.update(state, 0, reward, next_state, True)  # 动作任意，不影响
                break

            if done:
                # 步数耗尽，平局，无奖励
                reward = 0.0
                agent.update(state, 0, reward, next_state, True)
                break

            # 玩家2选择旋转
            action = agent.get_action(state)
            env.player2_rotate(action)

            # 更新（注意：当前动作的效果在下一个玩家1移动后才产生奖励）
            # 我们将在下一轮玩家1移动后获得奖励，所以这里先不更新，
            # 而是将 state 和 action 存起来，等下一轮状态出现后更新。
            # 调整逻辑：我们在本轮的玩家2动作之后，进入下一轮循环，
            # 当下一轮玩家1移动后得到 next_state 和 reward，再更新。
            # 所以需要保存当前 (state, action)
            last_state = state
            last_action = action

            # 现在状态变为玩家1移动之后的状态（在下一轮循环开始处处理）
            state = env._get_state()  # 当前已经是新的 step 状态

            # 这一轮的下一次玩家1移动将在下一轮循环执行，我们在此处无法获得奖励。
            # 因此需要重构循环，使用“保存上一轮的信息”方式。
            # 更清晰的做法是：在每一轮中，玩家2动作执行后，立刻进行玩家1移动（作为下一轮），
            # 这样奖励可以直接获得。让我们重写循环。

    # 注意到上面的循环逻辑有误，奖励传递不对，这里重新实现训练循环
    print("重新实现训练循环...")
    # 清空Q表重新训练
    agent.Q.clear()
    for ep in range(episodes):
        env.reset()
        state = env._get_state()  # (0,0,0,1)
        agent.epsilon = max(epsilon_end, epsilon_start - ep * epsilon_decay)
        done = False

        # 玩家1第一次移动
        dist = p1_move(env)
        state_after_p1, repeated, done = env.player1_move(dist)

        if repeated:
            reward = 1.0
            agent.update(state, 0, reward, state_after_p1, True)  # 其实动作无用
            continue
        if done:
            continue

        # 进入主循环
        while not done:
            # 现在 state_after_p1 是玩家1移动后的状态，玩家2要旋转
            s_current = state_after_p1
            action = agent.get_action(s_current)
            env.player2_rotate(action)

            # 玩家1下一次移动
            dist = p1_move(env)
            s_next, repeated, done = env.player1_move(dist)

            if repeated:
                reward = 1.0
                agent.update(s_current, action, reward, s_next, True)
                break
            if done:
                reward = 0.0
                agent.update(s_current, action, reward, s_next, True)
                break

            # 正常情况，奖励为0
            reward = 0.0
            agent.update(s_current, action, reward, s_next, False)
            state_after_p1 = s_next  # 下一个循环的状态

    return agent

# =================== 训练并展示策略 ===================
if __name__ == '__main__':
    print("开始训练 (使用随机玩家1，50000 episodes)...")
    agent = train(episodes=50000, p1_strategy='random')

    # # 提取学到的策略 (最优动作)
    # print("\n=== 学到的Q表 (部分展示) ===")
    # # 遍历所有可能状态
    # for sign_x in [0, 1]:
    #     for sign_y in [0, 1]:
    #         for d in range(4):
    #             for parity in [0, 1]:
    #                 f = (sign_x, sign_y, d, parity)
    #                 q = agent.Q[f]
    #                 best = 0 if q[0] > q[1] else 1
    #                 best_name = '左转' if best == 0 else '右转'
    #                 print(f"x>0:{sign_x}, y>0:{sign_y}, 朝向:{DIR_NAMES[d]}, 步奇偶:{parity} => {best_name} (Q:{q})")

    # 决策函数
    def predict_action(x, y, direction, step):
        """输入汽车坐标、朝向(0-3)和当前步数(第几次玩家1移动后，玩家2行动前)，返回建议动作字符串"""
        state = (x, y, direction, step)
        f = extract_features(state)
        q = agent.Q[f]
        best = 0 if q[0] >= q[1] else 1  # 平局时选左转
        return '左转' if best == 0 else '右转'

    print("\n=== 测试决策 ===")
    tests = [
        (6, 0, 0, 1),   # 奇数回合，y=0(不大于0)，应朝北(左转从东到北)
        (6, 2, 1, 2),   # 偶数回合，x=6>0，应朝西(从北左转是西)
        (-1, 2, 2, 3),  # 奇数回合，y=2>0，应朝南(从西左转是南)
        (5, -1, 0, 5),  # 奇数回合，y=-1<=0，应朝北(从东左转)
        (10, 3, 2, 12), # 偶数回合，x=10>0，应朝西(从西? 当前朝西，目标西，可能不需要转，但动作只能是左/右; 最好右转? 左转从西到南，右转到北。应选不改变? 但动作只有左右转，所以可能学到的策略会尽量选能达到目标的)
    ]
    for x, y, d, step in tests:
        act = predict_action(x, y, d, step)
        print(f"坐标({x},{y}), 朝向{DIR_NAMES[d]}, 步{step} -> {act}")

开始训练 (使用随机玩家1，50000 episodes)...
重新实现训练循环...

=== 测试决策 ===
坐标(6,0), 朝向东, 步1 -> 左转
坐标(6,2), 朝向北, 步2 -> 左转
坐标(-1,2), 朝向西, 步3 -> 左转
坐标(5,-1), 朝向东, 步5 -> 左转
坐标(10,3), 朝向西, 步12 -> 左转
